# 08 — Preparação para Streamlit

Neste notebook, vamos organizar os arquivos finais do projeto para a aplicação em Streamlit.

A modelagem já foi concluída. Agora o foco é transformar a análise em entrega.

Vamos preparar:

- série histórica;
- métricas dos modelos;
- previsões no período de teste;
- resumo do modelo final;
- arquivos padronizados para o app.

### 08.01 Imports e caminhos

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

import plotly.graph_objects as go

if Path.cwd().name == "notebooks":
    raiz_projeto = Path.cwd().parent
else:
    raiz_projeto = Path.cwd()

sys.path.append(str(raiz_projeto / "src"))

from visual_utils import (
    CORES,
    aplicar_layout_padrao,
    grafico_linha_padrao
)

caminho_outputs_tabelas = raiz_projeto / "outputs" / "tabelas"

caminho_serie_historica = caminho_outputs_tabelas / "serie_historica.csv"
caminho_metricas = caminho_outputs_tabelas / "metricas.csv"
caminho_previsoes_teste = caminho_outputs_tabelas / "previsoes_teste.csv"

print("Arquivos finais esperados:")
print("-", caminho_serie_historica)
print("-", caminho_metricas)
print("-", caminho_previsoes_teste)

Arquivos finais esperados:
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\serie_historica.csv
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\metricas.csv
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\previsoes_teste.csv


## Validando os arquivos finais

Antes de criar o app, precisamos garantir que os arquivos principais existem e podem ser carregados.

Essa etapa evita erros no Streamlit.

### 08.02 Verificar existência dos arquivos

In [2]:
arquivos_esperados = {
    "serie_historica": caminho_serie_historica,
    "metricas": caminho_metricas,
    "previsoes_teste": caminho_previsoes_teste
}

for nome, caminho in arquivos_esperados.items():
    if caminho.exists():
        print(f"OK - {nome}: {caminho}")
    else:
        print(f"ERRO - arquivo não encontrado: {caminho}")

OK - serie_historica: c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\serie_historica.csv
OK - metricas: c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\metricas.csv
OK - previsoes_teste: c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\previsoes_teste.csv


### 08.03 Carregar arquivos finais

In [3]:
serie_historica = pd.read_csv(caminho_serie_historica)
metricas = pd.read_csv(caminho_metricas)
previsoes_teste = pd.read_csv(caminho_previsoes_teste)

serie_historica["data_hora"] = pd.to_datetime(serie_historica["data_hora"])
previsoes_teste["data_hora"] = pd.to_datetime(previsoes_teste["data_hora"])

print("Série histórica:", serie_historica.shape)
print("Métricas:", metricas.shape)
print("Previsões no teste:", previsoes_teste.shape)

Série histórica: (456, 2)
Métricas: (4, 4)
Previsões no teste: (60, 6)


### 08.04 Conferir série histórica

In [4]:
serie_historica.head()

,data_hora,demanda
0,2011-01-01,985
1,2011-01-02,801
2,2011-01-03,1349
3,2011-01-04,1562
4,2011-01-05,1600


### 08.05 Conferir métricas

In [5]:
metricas

,modelo,MAE,RMSE,MAPE
0,"SARIMAX(1,1,1)(1,0,1,7)",908.079882,1092.154192,18.141702
1,Baseline,1585.183333,1865.908943,31.813723
2,"ARIMA(1,1,1)",1730.649703,2031.357449,34.679064
3,"SARIMA(1,1,1)(1,0,1,7)",1775.685477,2077.504788,35.530606


### 08.06 Conferir previsões

In [6]:
previsoes_teste.head()

,data_hora,demanda_real,previsao_baseline,previsao_arima,previsao_sarima,previsao_sarimax
0,2012-09-17,6869,7333,7486.108513,7486.740921,6900.850096
1,2012-09-18,4073,7333,7524.392858,7571.036681,6227.377995
2,2012-09-19,7591,7333,7533.965749,7587.662987,7208.963856
3,2012-10-01,6778,7333,7536.359422,7572.316395,7145.538681
4,2012-10-02,4639,7333,7536.957954,7590.249756,6900.933922


## Modelo final

Vamos identificar automaticamente o modelo com menor MAPE.

Esse será o modelo destacado na aplicação.

### 08.07 Identificar melhor modelo

In [7]:
metricas_ordenadas = metricas.sort_values("MAPE").reset_index(drop=True)

melhor_modelo = metricas_ordenadas.iloc[0]

nome_melhor_modelo = melhor_modelo["modelo"]
mae_melhor_modelo = melhor_modelo["MAE"]
rmse_melhor_modelo = melhor_modelo["RMSE"]
mape_melhor_modelo = melhor_modelo["MAPE"]

print("Melhor modelo:")
print(nome_melhor_modelo)

print("\nMétricas:")
print(f"MAE: {mae_melhor_modelo:.2f}")
print(f"RMSE: {rmse_melhor_modelo:.2f}")
print(f"MAPE: {mape_melhor_modelo:.2f}%")

Melhor modelo:
SARIMAX(1,1,1)(1,0,1,7)

Métricas:
MAE: 908.08
RMSE: 1092.15
MAPE: 18.14%


### 08.08 Criar resumo do projeto

In [8]:
resumo_projeto = pd.DataFrame({
    "item": [
        "modelo_final",
        "mae",
        "rmse",
        "mape",
        "inicio_serie",
        "fim_serie",
        "quantidade_dias_historico",
        "quantidade_dias_teste"
    ],
    "valor": [
        nome_melhor_modelo,
        round(mae_melhor_modelo, 2),
        round(rmse_melhor_modelo, 2),
        round(mape_melhor_modelo, 2),
        serie_historica["data_hora"].min(),
        serie_historica["data_hora"].max(),
        serie_historica.shape[0],
        previsoes_teste.shape[0]
    ]
})

resumo_projeto

,item,valor
0,modelo_final,"SARIMAX(1,1,1)(1,0,1,7)"
1,mae,908.08
2,rmse,1092.15
3,mape,18.14
4,inicio_serie,2011-01-01 00:00:00
5,fim_serie,2012-12-19 00:00:00
6,quantidade_dias_historico,456
7,quantidade_dias_teste,60


## Visualização final da série histórica

Este é o gráfico que pode ser usado no app para apresentar o comportamento geral da demanda.

### 08.09 Gráfico da série histórica

In [9]:
fig = grafico_linha_padrao(
    df=serie_historica,
    x="data_hora",
    y="demanda",
    titulo="Série histórica da demanda de bicicletas",
    labels={
        "data_hora": "Data",
        "demanda": "Demanda diária"
    },
    altura=500
)

fig.show()

## Visualização final das previsões

Agora vamos conferir o gráfico que compara o valor real com os modelos no período de teste.

### 08.10 Gráfico final de comparação

In [10]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["demanda_real"],
        mode="lines",
        name="Real",
        line=dict(color="#0B1F4D", width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_baseline"],
        mode="lines",
        name="Baseline",
        line=dict(color="#7A7A7A", width=2, dash="dash")
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_arima"],
        mode="lines",
        name="ARIMA",
        line=dict(color="#6FA8FF", width=2)
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_sarima"],
        mode="lines",
        name="SARIMA",
        line=dict(color="#F28E2B", width=2)
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_sarimax"],
        mode="lines",
        name="SARIMAX",
        line=dict(color="#2CA02C", width=3)
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Comparação das previsões no período de teste",
    altura=500
)

fig.show()

## Padronizando arquivos para o app

Para simplificar o Streamlit, vamos salvar os arquivos finais com nomes diretos:

- `serie_historica.csv`
- `metricas.csv`
- `previsoes.csv`
- `resumo_projeto.csv`

Todos ficarão dentro de `outputs/tabelas`.

### 08.11 Salvar arquivos padronizados

In [11]:
caminho_previsoes_app = caminho_outputs_tabelas / "previsoes.csv"
caminho_resumo_projeto = caminho_outputs_tabelas / "resumo_projeto.csv"

previsoes_teste.to_csv(
    caminho_previsoes_app,
    index=False,
    encoding="utf-8-sig"
)

resumo_projeto.to_csv(
    caminho_resumo_projeto,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos padronizados salvos:")
print("-", caminho_serie_historica)
print("-", caminho_metricas)
print("-", caminho_previsoes_app)
print("-", caminho_resumo_projeto)

Arquivos padronizados salvos:
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\serie_historica.csv
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\metricas.csv
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\previsoes.csv
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\resumo_projeto.csv


### 08.12 Validar arquivos do app

In [12]:
arquivos_app = [
    caminho_serie_historica,
    caminho_metricas,
    caminho_previsoes_app,
    caminho_resumo_projeto
]

for caminho in arquivos_app:
    df_teste = pd.read_csv(caminho)
    print(f"{caminho.name}: {df_teste.shape[0]} linhas e {df_teste.shape[1]} colunas")

serie_historica.csv: 456 linhas e 2 colunas
metricas.csv: 4 linhas e 4 colunas
previsoes.csv: 60 linhas e 6 colunas
resumo_projeto.csv: 8 linhas e 2 colunas


## Próximo passo

Os arquivos finais estão organizados para o Streamlit.

Na próxima etapa, vamos construir o `app.py` para apresentar:

- a série histórica;
- a comparação dos modelos;
- as previsões no período de teste;
- o resumo do modelo final.